In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from feature_engineering.order_flow_pipeline import OrderFlowFeatureEngineering
from feature_engineering.order_flow import L2QFeatureEngineering, TradeFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Test 1: Feature Engineering Only (should discover normalized files)
config_fe_only = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=None,  # Skip normalization
        feature_engineering=TradeFeatureEngineering(bar_duration_ms=BAR_DURATION_MS)
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=3,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=3,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

print("Testing Feature Engineering Only Pipeline...")
pipeline_fe = Pipeline(config_fe_only)
results_fe = pipeline_fe.run(slice(5))

In [ ]:
# Verify feature engineering results
if results_fe:
    res_pd = pd.DataFrame(results_fe)
    successful = res_pd.query("message == 'success'").shape[0]
    failed = res_pd.query("message != 'success'").shape[0]
    print(f"Feature Engineering Results: {successful} success, {failed} failed")
    
    # Show sample results
    for result in results_fe[:3]:
        if result['message'] == 'success':
            print(f"✓ {result.get('data_type', 'unknown')}: {result.get('row_count', 0):,} features -> {result['output_path']}")
        else:
            print(f"✗ Failed: {result['message'][:100]}")
else:
    print("No results returned")

In [ ]:
# Test 2: Full Pipeline (normalization + feature engineering)
config_full = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer(),
        feature_engineering=TradeFeatureEngineering(bar_duration_ms=BAR_DURATION_MS)
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir},
                  flat_core_count=3,
                  memory_multiplier=2.0,
                  memory_per_core_gb=4.0,
                  max_retries=3,
                  cpu_buffer=1,
                  file_sort_order="desc")
)

print("\nTesting Full Pipeline (Normalization + Feature Engineering)...")
pipeline_full = Pipeline(config_full)
results_full = pipeline_full.run(slice(3))

In [ ]:
# Verify full pipeline results
if results_full:
    print(f"\nFull Pipeline completed with {len(results_full)} results")
    
    # Show sample results
    for result in results_full[:3]:
        if result['message'] == 'success':
            print(f"✓ {result.get('data_type', 'unknown')}: {result.get('row_count', 0):,} features -> {result['output_path']}")
        else:
            print(f"✗ Failed: {result['message'][:100]}")
else:
    print("No results returned from full pipeline")